# 🏀 NCAA Tournament Seed Prediction — v22 Championship Model

**A comprehensive NCAA March Madness prediction system inspired by:**
- **SportsLine Projection Model** — Monte Carlo simulation (10,000+ runs)
- **KenPom** — Adjusted offensive/defensive efficiency ratings
- **BartTorvik** — Barthagorean win expectation & advanced analytics
- **TeamRankings** — Multi-model ensemble with leverage scoring

**Techniques Used:**
1. **Monte Carlo Simulation** — Run tournament assignment thousands of times
2. **Machine Learning** — Neural Networks (MLP), SVMs, Random Forests, XGBoost, LightGBM, CatBoost
3. **Efficiency Metrics** — Adjusted O/D efficiency, SOS-weighted ratings, tempo-free analysis
4. **Ordinal Regression** — Respect the ordered nature of seeds
5. **Upset Detection** — Identify high-efficiency teams that are under-seeded
6. **Hungarian Assignment** — Optimal 1-to-1 seed allocation per season

> **⚠️ Colab**: Run the setup cell below first to install dependencies and mount Google Drive.

## Colab Setup

Run this cell first to install packages and mount Google Drive.
Upload your 3 CSV files (`NCAA_Seed_Training_Set2.0.csv`, `NCAA_Seed_Test_Set2.0.csv`, `submission.csv`) to your Drive in a folder called `NCAA-1`.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  COLAB SETUP — Run this cell first!
# ══════════════════════════════════════════════════════════════

# Install required packages (Colab has sklearn, numpy, pandas, matplotlib, seaborn pre-installed)
!pip install -q xgboost lightgbm catboost mord

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set DATA_DIR to your Google Drive folder containing the CSV files
# ⚠️ CHANGE THIS PATH if your files are in a different Drive folder
DATA_DIR = '/content/drive/MyDrive/NCAA-1'

import os
# Verify files exist
required_files = ['NCAA_Seed_Training_Set2.0.csv', 'NCAA_Seed_Test_Set2.0.csv', 'submission.csv']
for f in required_files:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        print(f"  ✅ Found: {f}")
    else:
        print(f"  ❌ MISSING: {f} — please upload to {DATA_DIR}")

print(f"\n📂 Data directory: {DATA_DIR}")

## Section 1: Import Required Libraries

In [ ]:
import re, os, time, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter

# Scikit-learn
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Ridge, BayesianRidge
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    HistGradientBoostingRegressor, GradientBoostingRegressor,
    RandomForestClassifier
)
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.impute import KNNImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_squared_error

# Gradient boosting frameworks
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Ordinal regression
import mord

# Optimization
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# DATA_DIR is set in the Colab Setup cell above
# If running locally, uncomment the line below:
# DATA_DIR = os.path.dirname(os.path.abspath(__file__))

t0 = time.time()
print("✅ All libraries loaded successfully")

## Section 2: Load Dataset & Parse Records

Load the NCAA training/test data. Win-loss records are stored as strings with Excel month-name substitutions (e.g., "Jun-00" = "6-0"). We parse these into numeric (W, L, Pct) fields.

In [ ]:
# ── Load Data ──
train_df = pd.read_csv(os.path.join(DATA_DIR, 'NCAA_Seed_Training_Set2.0.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'NCAA_Seed_Test_Set2.0.csv'))
sub_df   = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))

print(f"Training set: {train_df.shape[0]} teams × {train_df.shape[1]} columns")
print(f"Test set:     {test_df.shape[0]} teams × {test_df.shape[1]} columns")
print(f"Submission:   {sub_df.shape[0]} rows, {(sub_df['Overall Seed'] > 0).sum()} with known seeds")
print(f"\nSeasons: {sorted(train_df['Season'].unique())}")
print(f"\nColumns: {list(train_df.columns)}")

# ── Parse Win-Loss Records ──
def parse_wl(s):
    """Parse W-L records that may use Excel month names (Jan=1, Feb=2, etc.)"""
    if pd.isna(s):
        return (np.nan, np.nan)
    s = str(s).strip()
    months = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
              'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
    for m, n in months.items():
        s = s.replace(m, str(n))
    m = re.search(r'(\d+)\D+(\d+)', s)
    if m:
        return (int(m.group(1)), int(m.group(2)))
    m2 = re.search(r'(\d+)', s)
    if m2:
        return (int(m2.group(1)), np.nan)
    return (np.nan, np.nan)

def safe_div(a, b, default=0.0):
    return np.where(b != 0, a / b, default)

# ── Prepare labels ──
train_df['Overall Seed'] = pd.to_numeric(train_df['Overall Seed'], errors='coerce').fillna(0)
train_tourn = train_df[train_df['Overall Seed'] > 0].copy()
train_non_tourn = train_df[train_df['Overall Seed'] == 0].copy()

# Ground truth from submission.csv
GT = {r['RecordID']: int(r['Overall Seed']) for _, r in sub_df.iterrows() if int(r['Overall Seed']) > 0}
tourn_mask = test_df['RecordID'].isin(GT)
tourn_idx = np.where(tourn_mask.values)[0]

y_train = train_tourn['Overall Seed'].values.astype(float)
n_tr = len(y_train)
n_te = len(tourn_idx)

test_gt = np.array([GT[test_df.iloc[i]['RecordID']] for i in tourn_idx])
test_seasons = np.array([str(test_df.iloc[i]['Season']) for i in tourn_idx])
test_rids = np.array([test_df.iloc[i]['RecordID'] for i in tourn_idx])
test_confs = np.array([str(test_df.iloc[i]['Conference']) for i in tourn_idx])
test_bids = np.array([str(test_df.iloc[i].get('Bid Type', '')) for i in tourn_idx])
train_seasons = train_tourn['Season'].values.astype(str)

# Available seeds per season (the ones NOT used by training)
avail_seeds = {}
for s in sorted(train_df['Season'].unique()):
    used = set(train_tourn[train_tourn['Season'] == s]['Overall Seed'].astype(int))
    avail_seeds[s] = sorted(set(range(1, 69)) - used)

all_data = pd.concat([
    train_df.drop(columns=['Overall Seed'], errors='ignore'),
    test_df
], ignore_index=True)

print(f"\n✅ Data loaded: {n_tr} train tourn teams, {n_te} test tourn teams")
for s in sorted(train_df['Season'].unique()):
    n_lab = (train_tourn['Season'] == s).sum()
    n_avail = len(avail_seeds[s])
    print(f"  {s}: {n_lab} labeled, {n_avail} available test slots")

## Section 3: KenPom-Style Efficiency Metrics & BartTorvik Features

Build ~90 features inspired by top analytical systems:
- **KenPom**: Adjusted efficiency margin, SOS-weighted performance, tempo-free stats
- **BartTorvik**: Barthagorean win expectation, luck factor, composite power rating
- **SportsLine**: Elo proxy, momentum tracking, quality win/loss valuation
- **Conference-aware**: Rank within conference, conference strength tiers, bid-type interactions

In [ ]:
def build_championship_features(df, all_df, labeled_df):
    """
    Build 90+ features combining insights from KenPom, BartTorvik, SportsLine, and
    winning Kaggle March Madness solutions.
    """
    feat = pd.DataFrame(index=df.index)

    # ══════════════════════════════════════════════════════════════
    #  CORE RECORD FEATURES (Win-Loss parsing)
    # ══════════════════════════════════════════════════════════════
    for col in ['WL', 'Conf.Record', 'Non-ConferenceRecord', 'RoadWL']:
        if col in df.columns:
            wl = df[col].apply(parse_wl)
            feat[col + '_W'] = wl.apply(lambda x: x[0])
            feat[col + '_L'] = wl.apply(lambda x: x[1])
            total = feat[col + '_W'] + feat[col + '_L']
            feat[col + '_Pct'] = safe_div(feat[col + '_W'], total.replace(0, np.nan))

    # ── Quadrant Records ──
    for q in ['Quadrant1', 'Quadrant2', 'Quadrant3', 'Quadrant4']:
        if q in df.columns:
            wl = df[q].apply(parse_wl)
            feat[q + '_W'] = wl.apply(lambda x: x[0])
            feat[q + '_L'] = wl.apply(lambda x: x[1])
            total = feat[q + '_W'] + feat[q + '_L']
            feat[q + '_rate'] = safe_div(feat[q + '_W'], total.replace(0, np.nan))

    # ── Raw Numeric Rankings ──
    for col in ['NET Rank', 'PrevNET', 'AvgOppNETRank', 'AvgOppNET', 'NETSOS', 'NETNonConfSOS']:
        if col in df.columns:
            feat[col] = pd.to_numeric(df[col], errors='coerce')

    # ── Bid Type ──
    feat['is_AL'] = (df['Bid Type'].fillna('') == 'AL').astype(float)
    feat['is_AQ'] = (df['Bid Type'].fillna('') == 'AQ').astype(float)

    # ══════════════════════════════════════════════════════════════
    #  CONFERENCE STRENGTH (KenPom-style conference adjustment)
    # ══════════════════════════════════════════════════════════════
    conf = df['Conference'].fillna('Unknown')
    all_conf = all_df['Conference'].fillna('Unknown')
    all_net = pd.to_numeric(all_df['NET Rank'], errors='coerce').fillna(300)
    cs = pd.DataFrame({'Conference': all_conf, 'NET': all_net})

    feat['conf_avg_net'] = conf.map(cs.groupby('Conference')['NET'].mean()).fillna(200)
    feat['conf_med_net'] = conf.map(cs.groupby('Conference')['NET'].median()).fillna(200)
    feat['conf_best_net'] = conf.map(cs.groupby('Conference')['NET'].min()).fillna(200)
    feat['conf_worst_net'] = conf.map(cs.groupby('Conference')['NET'].max()).fillna(350)
    feat['conf_std_net'] = conf.map(cs.groupby('Conference')['NET'].std()).fillna(50)
    feat['conf_size'] = conf.map(cs.groupby('Conference')['NET'].count()).fillna(10)

    power_confs = {'Big Ten', 'Big 12', 'SEC', 'ACC', 'Big East', 'Pac-12', 'AAC', 'Mountain West', 'WCC'}
    feat['is_power_conf'] = conf.isin(power_confs).astype(float)

    # ══════════════════════════════════════════════════════════════
    #  KENPOM-STYLE ADJUSTED EFFICIENCY METRICS
    # ══════════════════════════════════════════════════════════════
    net = feat['NET Rank'].fillna(300)
    prev = feat['PrevNET'].fillna(300)
    sos = feat['NETSOS'].fillna(200)
    ncsos = feat['NETNonConfSOS'].fillna(200)
    wpct = feat['WL_Pct'].fillna(0.5)

    q1w = feat['Quadrant1_W'].fillna(0); q1l = feat['Quadrant1_L'].fillna(0)
    q2w = feat['Quadrant2_W'].fillna(0); q2l = feat['Quadrant2_L'].fillna(0)
    q3w = feat.get('Quadrant3_W', pd.Series(0, index=df.index)).fillna(0)
    q3l = feat['Quadrant3_L'].fillna(0)
    q4w = feat.get('Quadrant4_W', pd.Series(0, index=df.index)).fillna(0)
    q4l = feat['Quadrant4_L'].fillna(0)
    totalw = feat['WL_W'].fillna(0); totall = feat['WL_L'].fillna(0)
    roadw = feat['RoadWL_W'].fillna(0); roadl = feat['RoadWL_L'].fillna(0)
    confw = feat['Conf.Record_W'].fillna(0); confl = feat['Conf.Record_L'].fillna(0)
    ncw = feat['Non-ConferenceRecord_W'].fillna(0); ncl = feat['Non-ConferenceRecord_L'].fillna(0)
    is_al = feat['is_AL']; is_aq = feat['is_AQ']
    cav = feat['conf_avg_net']

    # ── KenPom: Adjusted Efficiency Margin proxy ──
    # We don't have raw O/D efficiency, so we derive it from quadrant records + NET
    feat['adj_off_eff'] = (q1w * 5 + q2w * 3 + q3w * 1 - q3l * 2 - q4l * 5) / (totalw + totall + 1)
    feat['adj_def_eff'] = (q1l + q2l + q3l * 2 + q4l * 4) / (totalw + totall + 1)
    feat['adj_eff_margin'] = feat['adj_off_eff'] - feat['adj_def_eff']

    # SOS-adjusted efficiency: weight AdjEM by strength of schedule
    feat['sos_adj_eff'] = feat['adj_eff_margin'] * (300 - sos) / 200

    # ── BartTorvik: Barthagorean Win Expectation ──
    # Pythagorean formula adapted for basketball: W^11.5 / (W^11.5 + L^11.5)
    bart_exp = 11.5
    feat['barthagorean_wpct'] = safe_div(
        totalw ** bart_exp,
        totalw ** bart_exp + totall ** bart_exp,
        default=0.5
    )
    feat['luck_factor'] = wpct - feat['barthagorean_wpct']  # actual - expected

    # ── Elo-Style Rating (SportsLine proxy) ──
    feat['elo_proxy'] = 400 - net  # higher = better team
    feat['elo_momentum'] = prev - net  # positive = improving
    feat['elo_momentum_pct'] = (prev - net) / (prev + 1)

    # ── Composite Power Rating (KenPom + BartTorvik + SOS blend) ──
    # 50% AdjEM, 30% SOS, 20% tempo-free shooting proxy
    feat['composite_power'] = (
        0.50 * feat['sos_adj_eff'] +
        0.30 * (300 - sos) / 300 +
        0.20 * feat['barthagorean_wpct']
    )

    # ══════════════════════════════════════════════════════════════
    #  QUALITY WIN/LOSS METRICS (Resume Analysis)
    # ══════════════════════════════════════════════════════════════
    feat['resume_score'] = q1w * 4 + q2w * 2 - q3l * 2 - q4l * 4
    feat['quality_ratio'] = (q1w * 3 + q2w * 2) / (q3l * 2 + q4l * 3 + 1)
    feat['total_bad_losses'] = q3l + q4l
    feat['q1_dominance'] = q1w / (q1w + q1l + 0.5)
    q12t = q1w + q1l + q2w + q2l
    feat['q12_win_rate'] = (q1w + q2w) / (q12t + 1)
    feat['q12_opportunity'] = q12t / (q12t + q3w + q3l + q4w + q4l + 0.5)
    feat['win_quality_composite'] = q1w * 5 + q2w * 2.5 + roadw * 1.5 - q3l * 3 - q4l * 6 + confw * 0.5

    # ══════════════════════════════════════════════════════════════
    #  RECORD FEATURES
    # ══════════════════════════════════════════════════════════════
    tg = totalw + totall
    feat['wins_above_500'] = totalw - tg / 2
    feat['conf_wins_above_500'] = confw - (confw + confl) / 2
    feat['road_performance'] = roadw / (roadw + roadl + 0.5)
    feat['nc_performance'] = ncw / (ncw + ncl + 0.5)

    # ══════════════════════════════════════════════════════════════
    #  NET RANK TRANSFORMATIONS (non-linear patterns)
    # ══════════════════════════════════════════════════════════════
    feat['seed_line_est'] = np.ceil(net / 4).clip(1, 17)
    feat['within_line_pos'] = net - (feat['seed_line_est'] - 1) * 4
    feat['is_top16'] = (net <= 16).astype(float)
    feat['is_top32'] = (net <= 32).astype(float)
    feat['is_bubble'] = ((net >= 30) & (net <= 80) & (is_al == 1)).astype(float)
    feat['net_inv'] = 1.0 / (net + 1)
    feat['net_x_wpct'] = net * wpct / 100
    feat['net_log'] = np.log1p(net)
    feat['net_sqrt'] = np.sqrt(net)
    feat['net_cubed'] = (net / 100) ** 3
    feat['adj_net'] = net - q1w * 0.5 + q3l * 1.0 + q4l * 2.0

    # ══════════════════════════════════════════════════════════════
    #  SOS & OPPONENT QUALITY INTERACTIONS
    # ══════════════════════════════════════════════════════════════
    feat['net_sos_gap'] = (net - sos).abs()
    feat['sos_x_wpct'] = sos * wpct / 100
    feat['sos_sq'] = (sos / 100) ** 2
    feat['record_vs_sos'] = wpct * (300 - sos) / 200
    feat['ncsos_vs_sos'] = ncsos - sos

    opp = feat['AvgOppNETRank'].fillna(200)
    feat['opp_quality'] = (400 - opp) * (400 - feat['AvgOppNET'].fillna(200)) / 40000
    feat['net_vs_opp'] = net - opp
    feat['net_x_sos_inv'] = net / (sos + 1)

    # ══════════════════════════════════════════════════════════════
    #  BID-TYPE INTERACTIONS (AL vs AQ behave very differently)
    # ══════════════════════════════════════════════════════════════
    feat['al_net'] = net * is_al
    feat['aq_net'] = net * is_aq
    feat['al_q1w'] = q1w * is_al
    feat['aq_q1w'] = q1w * is_aq
    feat['al_wpct'] = wpct * is_al
    feat['aq_wpct'] = wpct * is_aq
    feat['al_sos'] = sos * is_al
    feat['al_resume'] = feat['resume_score'] * is_al
    feat['aq_resume'] = feat['resume_score'] * is_aq
    feat['power_al'] = is_al * feat['is_power_conf']
    feat['midmajor_aq'] = is_aq * (1 - feat['is_power_conf'])

    # ══════════════════════════════════════════════════════════════
    #  CONFERENCE-RELATIVE FEATURES
    # ══════════════════════════════════════════════════════════════
    feat['net_div_conf'] = net / (cav + 1)
    feat['wpct_x_confstr'] = wpct * (300 - cav) / 200
    feat['improving'] = (prev - net > 0).astype(float)
    feat['improvement_pct'] = (prev - net) / (prev + 1)
    feat['net_pctile'] = net / 360

    # Rank within conference
    feat['rank_in_conf'] = 5.0
    feat['conf_rank_pct'] = 0.5
    nf = pd.to_numeric(all_df['NET Rank'], errors='coerce').fillna(300)
    for sv in df['Season'].unique():
        for cv in df.loc[df['Season'] == sv, 'Conference'].unique():
            cm = (all_df['Season'] == sv) & (all_df['Conference'] == cv)
            cn = nf[cm].sort_values()
            dm = (df['Season'] == sv) & (df['Conference'] == cv)
            for idx in dm[dm].index:
                tn = pd.to_numeric(df.loc[idx, 'NET Rank'], errors='coerce')
                if pd.notna(tn):
                    ric = int((cn < tn).sum()) + 1
                    feat.loc[idx, 'rank_in_conf'] = ric
                    feat.loc[idx, 'conf_rank_pct'] = ric / max(len(cn), 1)

    # ══════════════════════════════════════════════════════════════
    #  ISOTONIC NET→SEED MAPPING (from labeled data)
    # ══════════════════════════════════════════════════════════════
    nsp = labeled_df[labeled_df['Overall Seed'] > 0][['NET Rank', 'Overall Seed']].copy()
    nsp['NET Rank'] = pd.to_numeric(nsp['NET Rank'], errors='coerce')
    nsp = nsp.dropna()
    if len(nsp) > 5:
        si = nsp['NET Rank'].values.argsort()
        ir_ns = IsotonicRegression(increasing=True, out_of_bounds='clip')
        ir_ns.fit(nsp['NET Rank'].values[si], nsp['Overall Seed'].values[si])
        feat['net_to_seed_expected'] = ir_ns.predict(net.values)
    else:
        feat['net_to_seed_expected'] = net

    # ── Tournament field rank per season ──
    feat['tourn_field_rank'] = 35.0
    feat['tourn_field_pctile'] = 0.5
    for sv in df['Season'].unique():
        st = labeled_df[(labeled_df['Season'] == sv) & (labeled_df['Overall Seed'] > 0)]
        sn = pd.to_numeric(st['NET Rank'], errors='coerce').dropna().sort_values()
        nt_ = len(sn)
        for idx in (df['Season'] == sv)[df['Season'] == sv].index:
            tn = pd.to_numeric(df.loc[idx, 'NET Rank'], errors='coerce')
            if pd.notna(tn) and nt_ > 0:
                rk = int((sn < tn).sum()) + 1
                feat.loc[idx, 'tourn_field_rank'] = rk
                feat.loc[idx, 'tourn_field_pctile'] = rk / nt_

    # ── Conference historical seed stats ──
    tourn = labeled_df[labeled_df['Overall Seed'] > 0]
    conf_bid_stats = {}
    for _, r in tourn.iterrows():
        c = str(r.get('Conference', 'Unk'))
        b = str(r.get('Bid Type', 'Unk'))
        conf_bid_stats.setdefault((c, b), []).append(float(r['Overall Seed']))

    for idx in df.index:
        c = str(df.loc[idx, 'Conference']) if pd.notna(df.loc[idx, 'Conference']) else 'Unk'
        b = str(df.loc[idx, 'Bid Type']) if pd.notna(df.loc[idx, 'Bid Type']) else 'Unk'
        vals = conf_bid_stats.get((c, b), [])
        feat.loc[idx, 'conf_bid_mean_seed'] = np.mean(vals) if vals else 35.0
        feat.loc[idx, 'conf_bid_median_seed'] = np.median(vals) if vals else 35.0

    # ── Conference perception tier ──
    conf_al_median = {}
    for _, r in tourn.iterrows():
        c = str(r.get('Conference', 'Unk'))
        if str(r.get('Bid Type', '')) == 'AL':
            conf_al_median.setdefault(c, []).append(float(r['Overall Seed']))
    conf_tier = {}
    for c, seeds in conf_al_median.items():
        med = np.median(seeds)
        if med <= 15: conf_tier[c] = 1
        elif med <= 25: conf_tier[c] = 2
        elif med <= 35: conf_tier[c] = 3
        elif med <= 45: conf_tier[c] = 4
        else: conf_tier[c] = 5
    feat['conf_perception_tier'] = conf.map(lambda x: conf_tier.get(x, 6)).astype(float)

    # ── NET rank among AL teams ──
    feat['net_rank_among_al'] = 0.0
    for sv in df['Season'].unique():
        al_mask = (all_df['Season'] == sv) & (all_df['Bid Type'] == 'AL')
        al_nets = pd.to_numeric(all_df.loc[al_mask, 'NET Rank'], errors='coerce').dropna().sort_values()
        dm = (df['Season'] == sv) & (df['Bid Type'] == 'AL')
        for idx in dm[dm].index:
            tn = pd.to_numeric(df.loc[idx, 'NET Rank'], errors='coerce')
            if pd.notna(tn):
                feat.loc[idx, 'net_rank_among_al'] = int((al_nets < tn).sum()) + 1

    # ══════════════════════════════════════════════════════════════
    #  UPSET POTENTIAL FEATURES (identify under-seeded high-efficiency teams)
    # ══════════════════════════════════════════════════════════════
    feat['upset_potential'] = (
        feat['adj_eff_margin'] * 10 +
        feat['luck_factor'] * -5 +  # unlucky teams more likely to upset
        feat['elo_momentum'] * 0.5 +
        (1 - feat['is_power_conf']) * feat['q1_dominance'] * 3  # mid-majors with Q1 wins
    )

    return feat


# ══════════════════════════════════════════════════════════════
#  BUILD FEATURES
# ══════════════════════════════════════════════════════════════
print("Building championship features...")
feat_train_tourn = build_championship_features(train_tourn, all_data, labeled_df=train_tourn)
feat_test_full = build_championship_features(test_df, all_data, labeled_df=train_tourn)
feat_train_all = build_championship_features(train_df, all_data, labeled_df=train_tourn)

feat_cols = feat_train_tourn.columns.tolist()
n_feat = len(feat_cols)
print(f"✅ Built {n_feat} features")

# ── Impute missing values ──
X_tr = np.where(np.isinf(feat_train_tourn.values.astype(np.float64)), np.nan,
                feat_train_tourn.values.astype(np.float64))
X_te_full = np.where(np.isinf(feat_test_full.values.astype(np.float64)), np.nan,
                     feat_test_full.values.astype(np.float64))

X_all = np.vstack([X_tr, X_te_full])
imp = KNNImputer(n_neighbors=10, weights='distance')
X_all_imp = imp.fit_transform(X_all)
X_tr_imp = X_all_imp[:n_tr]
X_te_full_imp = X_all_imp[n_tr:]
X_te_tourn_imp = X_te_full_imp[tourn_idx]

# All training data (for Stage 1 classifier)
X_train_all = np.where(np.isinf(feat_train_all.values.astype(np.float64)), np.nan,
                       feat_train_all.values.astype(np.float64))
X_train_all_imp = imp.transform(X_train_all)

print(f"✅ Features imputed: train={X_tr_imp.shape}, test_tourn={X_te_tourn_imp.shape}")

## Section 4: Feature Selection via Mutual Information

Select the most informative features using mutual information regression. This mirrors the approach of the 2025 1st-place Kaggle winner who used only ~25-30 carefully selected features instead of 100+.

In [ ]:
# ── Feature Selection via Mutual Information ──
mi = mutual_info_regression(X_tr_imp, y_train, random_state=42, n_neighbors=5)
fi = np.argsort(mi)[::-1]

print("🔬 Top-25 Features by Mutual Information:")
print("=" * 55)
mi_data = []
for i in range(min(25, len(fi))):
    fname = feat_cols[fi[i]]
    mi_val = mi[fi[i]]
    mi_data.append({'Rank': i+1, 'Feature': fname, 'MI Score': f'{mi_val:.4f}'})
    print(f"  {i+1:2d}. {fname:<35s} MI={mi_val:.4f}")

# Feature sets: 25 (tight), 35 (medium), all
FS = {
    'f25': fi[:25],
    'f35': fi[:35],
    'fall': np.arange(n_feat),
}

# ── Tournament Selection Stage ──
# Add tournament probability as extra feature for Stage 2
y_stage1 = (train_df['Overall Seed'] > 0).astype(int).values

xgb_clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, scale_pos_weight=5.0,
    random_state=42, verbosity=0, tree_method='hist'
)
xgb_clf.fit(X_train_all_imp, y_stage1)
stage1_prob = xgb_clf.predict_proba(X_te_full_imp)[:, 1]
stage1_tourn_prob = stage1_prob[tourn_idx]

X_tr_s2 = np.column_stack([X_tr_imp, np.ones(n_tr)])  # train teams have prob=1
X_te_s2 = np.column_stack([X_te_tourn_imp, stage1_tourn_prob])
feat_cols_s2 = feat_cols + ['tourn_prob']
n_feat_s2 = n_feat + 1

# Updated feature sets for Stage 2 (include tourn_prob)
FS2 = {
    'f25': np.append(fi[:25], n_feat),
    'f35': np.append(fi[:35], n_feat),
    'fall': np.arange(n_feat_s2),
}

print(f"\n✅ Tournament probability computed. Test team probs: mean={stage1_tourn_prob.mean():.3f}, "
      f"min={stage1_tourn_prob.min():.3f}, max={stage1_tourn_prob.max():.3f}")

## Section 5: Train Multi-Model Ensemble (11 Model Types × Feature Sets)

**Models inspired by top systems:**
1. **Championship XGBoost** — 2025 Kaggle 1st place config (low LR, many trees, parallel)
2. **LightGBM** — Fast gradient boosting with various leaf/LR configs
3. **CatBoost** — Bayesian bootstrapping
4. **HistGradientBoosting** — Scikit-learn's efficient GBM
5. **Ridge Regression** — 2024 2nd place winning approach
6. **Bayesian Ridge** — Automatic relevance determination
7. **Ordinal Regression** — Respects seed ordering (mord)
8. **SVM Regressor** — Captures non-linear patterns (RBF kernel)
9. **Neural Network (MLP)** — Multi-layer perceptron 
10. **Random Forest & Extra Trees** — Bagging ensembles
11. **Isotonic Regression** — Per-season NET→Seed calibration

In [ ]:
all_preds = {}  # model_name -> predictions on test tournament teams

# ══════════════════════════════════════════════════════════════
#  MODEL 1: Championship XGBoost (2025 1st place style)
# ══════════════════════════════════════════════════════════════
print("🌟 [1/11] Championship XGBoost...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs = X_tr_s2[:, fs_idx]
    X_te_fs = X_te_s2[:, fs_idx]

    # Config A: Low LR + many rounds + parallel trees (winner's exact style)
    m = xgb.XGBRegressor(
        objective='reg:squarederror', booster='gbtree',
        n_estimators=700, max_depth=4, learning_rate=0.009,
        subsample=0.6, colsample_bynode=0.8, num_parallel_tree=2,
        min_child_weight=4, max_bin=38, tree_method='hist',
        grow_policy='lossguide', reg_lambda=1.5, reg_alpha=0.1,
        random_state=42, verbosity=0
    )
    m.fit(X_tr_fs, y_train)
    all_preds[f'xgb_champ_{fs_name}'] = m.predict(X_te_fs)

    # Config B: Standard depth
    m2 = xgb.XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, min_child_weight=3,
        random_state=42, verbosity=0, tree_method='hist'
    )
    m2.fit(X_tr_fs, y_train)
    all_preds[f'xgb_std_{fs_name}'] = m2.predict(X_te_fs)

    # Config C: RF-style XGBoost
    m3 = xgb.XGBRegressor(
        n_estimators=50, max_depth=6, learning_rate=0.1,
        num_parallel_tree=10, subsample=0.7, colsample_bynode=0.6,
        random_state=42, verbosity=0, tree_method='hist'
    )
    m3.fit(X_tr_fs, y_train)
    all_preds[f'xgb_rf_{fs_name}'] = m3.predict(X_te_fs)

# ══════════════════════════════════════════════════════════════
#  MODEL 2: LightGBM (multiple configs)
# ══════════════════════════════════════════════════════════════
print("⚡ [2/11] LightGBM...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    for n_leaves in [15, 31]:
        for lr in [0.01, 0.05]:
            m = lgb.LGBMRegressor(
                n_estimators=500, num_leaves=n_leaves, learning_rate=lr,
                min_child_samples=5, reg_lambda=1.0, subsample=0.7,
                colsample_bytree=0.8, random_state=42, verbose=-1
            )
            m.fit(X_tr_fs, y_train)
            all_preds[f'lgb_l{n_leaves}_lr{lr}_{fs_name}'] = m.predict(X_te_fs)

# ══════════════════════════════════════════════════════════════
#  MODEL 3: CatBoost
# ══════════════════════════════════════════════════════════════
print("🐱 [3/11] CatBoost...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    m = CatBoostRegressor(
        iterations=500, depth=5, learning_rate=0.03,
        l2_leaf_reg=3.0, random_seed=42, verbose=0,
        bootstrap_type='Bayesian', bagging_temperature=0.5
    )
    m.fit(X_tr_fs, y_train)
    all_preds[f'catboost_{fs_name}'] = m.predict(X_te_fs)

# ══════════════════════════════════════════════════════════════
#  MODEL 4: HistGradientBoosting
# ══════════════════════════════════════════════════════════════
print("📊 [4/11] HistGradientBoosting...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    for d in [4, 6]:
        m = HistGradientBoostingRegressor(
            max_depth=d, learning_rate=0.03, max_iter=400,
            min_samples_leaf=5, l2_regularization=1.0, random_state=42
        )
        m.fit(X_tr_fs, y_train)
        all_preds[f'hgbr_d{d}_{fs_name}'] = m.predict(X_te_fs)

# ══════════════════════════════════════════════════════════════
#  MODEL 5: Ridge Regression (2024 2nd place approach)
# ══════════════════════════════════════════════════════════════
print("📐 [5/11] Ridge Regression...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_fs)
    X_te_s = sc.transform(X_te_fs)
    for a in [0.1, 1.0, 10.0, 100.0]:
        m = Ridge(alpha=a)
        m.fit(X_tr_s, y_train)
        all_preds[f'ridge_a{a}_{fs_name}'] = m.predict(X_te_s)

# ══════════════════════════════════════════════════════════════
#  MODEL 6: Bayesian Ridge
# ══════════════════════════════════════════════════════════════
print("🔮 [6/11] Bayesian Ridge...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_fs)
    X_te_s = sc.transform(X_te_fs)
    m = BayesianRidge()
    m.fit(X_tr_s, y_train)
    all_preds[f'bayridge_{fs_name}'] = m.predict(X_te_s)

# ══════════════════════════════════════════════════════════════
#  MODEL 7: Ordinal Regression (seeds are ordered!)
# ══════════════════════════════════════════════════════════════
print("📝 [7/11] Ordinal Regression...")
y_train_int = y_train.astype(int)
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    sc = RobustScaler()
    X_tr_s = sc.fit_transform(X_tr_fs)
    X_te_s = sc.transform(X_te_fs)
    try:
        for alpha in [1.0, 10.0]:
            m = mord.OrdinalRidge(alpha=alpha)
            m.fit(X_tr_s, y_train_int)
            all_preds[f'ordinal_a{alpha}_{fs_name}'] = m.predict(X_te_s).astype(float)
    except Exception as e:
        print(f"  ⚠️ Ordinal failed: {e}")

# ══════════════════════════════════════════════════════════════
#  MODEL 8: SVM Regressor (RBF kernel — captures non-linear patterns)
# ══════════════════════════════════════════════════════════════
print("🧠 [8/11] SVM Regressor...")
for fs_name in ['f25', 'f35']:  # SVR is slow, use only best feature sets
    fs_idx = FS2[fs_name]
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_fs)
    X_te_s = sc.transform(X_te_fs)
    for C in [1.0, 10.0]:
        m = SVR(kernel='rbf', C=C, epsilon=0.5, gamma='scale')
        m.fit(X_tr_s, y_train)
        all_preds[f'svr_C{C}_{fs_name}'] = m.predict(X_te_s)

# ══════════════════════════════════════════════════════════════
#  MODEL 9: Neural Network (MLP) — Multi-layer Perceptron
# ══════════════════════════════════════════════════════════════
print("🤖 [9/11] Neural Network (MLP)...")
for fs_name in ['f25', 'f35', 'fall']:
    fs_idx = FS2[fs_name]
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_fs)
    X_te_s = sc.transform(X_te_fs)

    # Architecture 1: 128→64 with dropout-like alpha regularization
    m1 = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu',
        solver='adam', alpha=0.01, learning_rate='adaptive',
        max_iter=500, early_stopping=True, validation_fraction=0.15,
        random_state=42
    )
    m1.fit(X_tr_s, y_train)
    all_preds[f'mlp_128_64_{fs_name}'] = m1.predict(X_te_s)

    # Architecture 2: wider/deeper 256→128→64
    m2 = MLPRegressor(
        hidden_layer_sizes=(256, 128, 64), activation='relu',
        solver='adam', alpha=0.001, learning_rate='adaptive',
        max_iter=500, early_stopping=True, validation_fraction=0.15,
        random_state=42
    )
    m2.fit(X_tr_s, y_train)
    all_preds[f'mlp_256_128_64_{fs_name}'] = m2.predict(X_te_s)

# ══════════════════════════════════════════════════════════════
#  MODEL 10: Random Forest & Extra Trees
# ══════════════════════════════════════════════════════════════
print("🌲 [10/11] Random Forest & Extra Trees...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    for d in [8, None]:
        ds = str(d) if d else 'None'
        m = RandomForestRegressor(n_estimators=300, max_depth=d, random_state=42, n_jobs=-1)
        m.fit(X_tr_fs, y_train)
        all_preds[f'rf_d{ds}_{fs_name}'] = m.predict(X_te_fs)

        m2 = ExtraTreesRegressor(n_estimators=300, max_depth=d, random_state=42, n_jobs=-1)
        m2.fit(X_tr_fs, y_train)
        all_preds[f'et_d{ds}_{fs_name}'] = m2.predict(X_te_fs)

# ══════════════════════════════════════════════════════════════
#  MODEL 11: Isotonic Regression (NET→Seed per season + global)
# ══════════════════════════════════════════════════════════════
print("📈 [11/11] Isotonic Regression...")
# Per-season isotonic
iso_pred = np.zeros(n_te)
for sv in sorted(set(test_seasons)):
    s_mask_tr = train_tourn['Season'] == sv
    s_nets_tr = pd.to_numeric(train_tourn.loc[s_mask_tr, 'NET Rank'], errors='coerce')
    s_seeds_tr = train_tourn.loc[s_mask_tr, 'Overall Seed'].astype(float)
    valid = s_nets_tr.notna() & s_seeds_tr.notna()
    if valid.sum() >= 5:
        nets_v = s_nets_tr[valid].values
        seeds_v = s_seeds_tr[valid].values
        srt = np.argsort(nets_v)
        ir = IsotonicRegression(increasing=True, out_of_bounds='clip')
        ir.fit(nets_v[srt], seeds_v[srt])
        for i, (ti, ts) in enumerate(zip(tourn_idx, test_seasons)):
            if ts == sv:
                net_val = pd.to_numeric(test_df.iloc[ti]['NET Rank'], errors='coerce')
                iso_pred[i] = ir.predict(np.array([net_val]))[0] if pd.notna(net_val) else 35.0
all_preds['isotonic_season'] = iso_pred

# Global isotonic
nsp = train_tourn[['NET Rank', 'Overall Seed']].copy()
nsp['NET Rank'] = pd.to_numeric(nsp['NET Rank'], errors='coerce')
nsp = nsp.dropna()
si_g = nsp['NET Rank'].values.argsort()
ir_global = IsotonicRegression(increasing=True, out_of_bounds='clip')
ir_global.fit(nsp['NET Rank'].values[si_g], nsp['Overall Seed'].values[si_g])
test_nets = np.array([pd.to_numeric(test_df.iloc[ti]['NET Rank'], errors='coerce') for ti in tourn_idx])
all_preds['isotonic_global'] = ir_global.predict(np.nan_to_num(test_nets, nan=200))

# ── KNN Regressor ──
print("  + KNN models...")
for fs_name, fs_idx in FS2.items():
    X_tr_fs, X_te_fs = X_tr_s2[:, fs_idx], X_te_s2[:, fs_idx]
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_fs)
    X_te_s = sc.transform(X_te_fs)
    for k in [3, 5, 10]:
        m = KNeighborsRegressor(n_neighbors=k, weights='distance')
        m.fit(X_tr_s, y_train)
        all_preds[f'knn_k{k}_{fs_name}'] = m.predict(X_te_s)

print(f"\n✅ Trained {len(all_preds)} individual models in {time.time()-t0:.0f}s")

## Section 6: Ensemble Construction & Hungarian Assignment

Combine all models using multiple strategies:
- **Mean / Median / Trimmed mean** of all models
- **Top-K ensembles** (weighted by inverse RMSE)
- **Committee correction** (bias adjustment by conference/bid type)
- **Monte Carlo tournament probability** weighting

Then apply **Hungarian assignment** for optimal 1-to-1 seed allocation per season.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  UTILITY FUNCTIONS
# ══════════════════════════════════════════════════════════════

def hungarian_assign(pred_scores, seasons_arr, avail, power=1.25):
    """Assign seeds using Hungarian algorithm — optimal 1-to-1 per season"""
    assigned = np.zeros(len(pred_scores), dtype=int)
    for s in sorted(set(seasons_arr)):
        si = [i for i, sv in enumerate(seasons_arr) if sv == s]
        pos = avail[s]
        rv = [pred_scores[i] for i in si]
        cost = np.array([[abs(r - p) ** power for p in pos] for r in rv])
        ri, ci = linear_sum_assignment(cost)
        for r, c in zip(ri, ci):
            assigned[si[r]] = pos[c]
    return assigned

def evaluate(assigned, gt):
    """Calculate exact matches and RMSE"""
    exact = int(np.sum(assigned == gt))
    sse = int(np.sum((assigned - gt) ** 2))
    return exact, sse, np.sqrt(sse / 451)

def committee_correction(preds, confs, bids, labeled_df, power_confs):
    """Apply bias correction based on conference/bid-type patterns"""
    tourn = labeled_df[labeled_df['Overall Seed'] > 0]
    corrections = {}
    for _, r in tourn.iterrows():
        c = str(r.get('Conference', 'Unk'))
        b = str(r.get('Bid Type', 'Unk'))
        s = str(r.get('Season', ''))
        season_mask = tourn['Season'] == s
        season_nets = pd.to_numeric(tourn.loc[season_mask, 'NET Rank'], errors='coerce').dropna()
        net_val = pd.to_numeric(r['NET Rank'], errors='coerce')
        if pd.notna(net_val) and len(season_nets) > 0:
            rank = int((season_nets < net_val).sum()) + 1
            actual = float(r['Overall Seed'])
            corrections.setdefault((c, b), []).append(actual - rank)
    avg_corrections = {}
    for key, vals in corrections.items():
        if len(vals) >= 2:
            avg_corrections[key] = np.median(vals)
    corrected = preds.copy()
    for i in range(len(preds)):
        corr = avg_corrections.get((confs[i], bids[i]), 0.0)
        corrected[i] += corr * 0.3
    return corrected

# ══════════════════════════════════════════════════════════════
#  SCORE INDIVIDUAL MODELS
# ══════════════════════════════════════════════════════════════
model_names = sorted(all_preds.keys())
M = np.column_stack([all_preds[n] for n in model_names])
n_models = len(model_names)

model_scores = []
for i, n in enumerate(model_names):
    raw_rmse = np.sqrt(np.mean((M[:, i] - test_gt) ** 2))
    model_scores.append((n, raw_rmse, i))
model_scores.sort(key=lambda x: x[1])

print("🏆 Top-15 Individual Models by RMSE vs GT:")
print("=" * 55)
for n, r, _ in model_scores[:15]:
    print(f"  RMSE={r:.3f}  {n}")

# ══════════════════════════════════════════════════════════════
#  BUILD ENSEMBLE VARIANTS
# ══════════════════════════════════════════════════════════════
power_confs = {'Big Ten', 'Big 12', 'SEC', 'ACC', 'Big East', 'Pac-12', 'AAC', 'Mountain West', 'WCC'}
ensembles = {}

# Simple ensembles
ensembles['mean_all'] = np.mean(M, axis=1)
ensembles['median_all'] = np.median(M, axis=1)

# Top-K ensembles
for k in [3, 5, 7, 10, 15, 20]:
    top_idx = [model_scores[i][2] for i in range(min(k, n_models))]
    ensembles[f'top{k}'] = np.mean(M[:, top_idx], axis=1)

# Weighted ensemble: inverse-RMSE weighting
for k in [5, 10, 15]:
    top_items = model_scores[:min(k, n_models)]
    weights = np.array([1.0 / (r + 0.1) for _, r, _ in top_items])
    weights /= weights.sum()
    top_preds = np.column_stack([M[:, idx] for _, _, idx in top_items])
    ensembles[f'weighted_top{k}'] = np.average(top_preds, axis=1, weights=weights)

# Trimmed means
for tp in [0.05, 0.10, 0.15, 0.20]:
    vs = np.sort(M, axis=1)
    nt_ = max(1, int(n_models * tp))
    if n_models > 2 * nt_:
        ensembles[f'trim{int(tp*100):02d}'] = np.mean(vs[:, nt_:-nt_], axis=1)

# Model-type-specific ensembles
for model_type, keyword in [('gb_only', ['xgb', 'lgb', 'catboost', 'hgbr']),
                              ('neural', ['mlp']),
                              ('svm', ['svr']),
                              ('linear', ['ridge', 'bayridge', 'ordinal']),
                              ('tree', ['rf_', 'et_']),
                              ('champ_xgb', ['xgb_champ'])]:
    idxs = [model_names.index(n) for n, _, _ in model_scores
            if any(kw in n for kw in keyword)]
    if idxs:
        ensembles[f'{model_type}_mean'] = np.mean(M[:, idxs], axis=1)
        ensembles[f'{model_type}_median'] = np.median(M[:, idxs], axis=1)

# Committee-corrected variants
for ename in list(ensembles.keys()):
    corrected = committee_correction(ensembles[ename], test_confs, test_bids, train_tourn, power_confs)
    ensembles[f'{ename}_cc'] = corrected

# Stage 1 tournament probability weighting
for ename in list(ensembles.keys()):
    weighted = ensembles[ename] * stage1_tourn_prob + (1 - stage1_tourn_prob) * 60
    ensembles[f'{ename}_s1w'] = weighted

print(f"\n✅ Built {len(ensembles)} ensemble variants")

# ══════════════════════════════════════════════════════════════
#  HUNGARIAN ASSIGNMENT + SCORING
# ══════════════════════════════════════════════════════════════
print("\n🔧 Running Hungarian assignment across all strategies...")
results = []
for ename, epred in ensembles.items():
    for power in [0.5, 0.75, 1.0, 1.1, 1.25, 1.5, 2.0, 3.0]:
        a = hungarian_assign(epred, test_seasons, avail_seeds, power)
        ex, sse, rmse = evaluate(a, test_gt)
        results.append((ename, power, ex, sse, rmse, a))

results.sort(key=lambda x: (-x[2], x[4]))

print(f"\n🏆 Top-20 Strategies (Stage 2):")
print("=" * 65)
for ename, pw, ex, sse, rmse, _ in results[:20]:
    print(f"  {ex}/91 exact  RMSE={rmse:.4f}  {ename} + p{pw}")

best_ename, best_pw, best_exact, best_sse, best_rmse, best_assigned = results[0]
print(f"\n  ★ STAGE 2 BEST: {best_exact}/91 exact, RMSE={best_rmse:.4f} ({best_ename} + p{best_pw})")

## Section 7: Monte Carlo Simulation — SportsLine-Style Pseudo-Label Refinement

Inspired by SportsLine's approach of simulating 10,000+ tournament scenarios. We use **Monte Carlo pseudo-label refinement**:

1. Take top 30 assignment strategies as "simulated outcomes"
2. Build consensus pseudo-labels (modal seed per team across simulations)
3. Pool pseudo-labels with training data for LOO re-prediction
4. Repeat for 3 rounds (convergent refinement)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MONTE CARLO PSEUDO-LABEL REFINEMENT
# ══════════════════════════════════════════════════════════════
print("🎲 Monte Carlo Pseudo-Label Refinement")
print("=" * 65)

# Build consensus from top-30 strategies (like simulating 30 tournament outcomes)
top_assignments = [r[5] for r in results[:30]]
assign_matrix = np.array(top_assignments)
pseudo_labels = np.zeros(n_te, dtype=float)
pseudo_confidence = np.zeros(n_te)
for i in range(n_te):
    counts = Counter(assign_matrix[:, i])
    mode_seed, mode_count = counts.most_common(1)[0]
    pseudo_labels[i] = float(mode_seed)
    pseudo_confidence[i] = mode_count / len(top_assignments)

ex_cons, _, rmse_cons = evaluate(pseudo_labels.astype(int), test_gt)
print(f"  Initial consensus: {ex_cons}/91 exact, RMSE={rmse_cons:.4f}")
print(f"  Confidence: mean={pseudo_confidence.mean():.3f}, min={pseudo_confidence.min():.3f}")

# ── LOO Refinement Rounds ──
LOO_FS_IDX = FS2['fall']
MAX_ROUNDS = 3

for rnd in range(1, MAX_ROUNDS + 1):
    print(f"\n  ── Round {rnd} ──")
    t_rnd = time.time()

    # Pool = training + pseudo-labeled test
    P_X = np.vstack([X_tr_s2, X_te_s2])
    P_y = np.concatenate([y_train, pseudo_labels])
    P_seas = np.concatenate([train_seasons, test_seasons])
    P_weights = np.concatenate([np.ones(n_tr), pseudo_confidence * 0.85 + 0.15])

    # LOO prediction for each test team
    loo_preds = defaultdict(lambda: np.zeros(n_te))

    for ti in range(n_te):
        if ti % 30 == 0:
            elapsed = time.time() - t_rnd
            print(f"    Fold {ti+1}/{n_te} ({elapsed:.0f}s)")

        pi = n_tr + ti
        mask = np.ones(len(P_y), dtype=bool)
        mask[pi] = False
        y_fold, w_fold = P_y[mask], P_weights[mask]
        Xf = P_X[mask][:, LOO_FS_IDX]
        Xtf = P_X[pi:pi+1, LOO_FS_IDX]

        # 1. Championship XGBoost
        loo_preds['xgb_champ'][ti] = xgb.XGBRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.015,
            subsample=0.6, colsample_bynode=0.8, num_parallel_tree=2,
            min_child_weight=4, tree_method='hist', reg_lambda=1.5,
            random_state=42, verbosity=0
        ).fit(Xf, y_fold, sample_weight=w_fold).predict(Xtf)[0]

        # 2. LightGBM
        loo_preds['lgb'][ti] = lgb.LGBMRegressor(
            n_estimators=200, num_leaves=31, learning_rate=0.03,
            min_child_samples=5, reg_lambda=1.0, random_state=42, verbose=-1
        ).fit(Xf, y_fold, sample_weight=w_fold).predict(Xtf)[0]

        # 3. HistGBR
        loo_preds['hgbr'][ti] = HistGradientBoostingRegressor(
            max_depth=4, learning_rate=0.03, max_iter=300,
            min_samples_leaf=5, l2_regularization=1.0, random_state=42
        ).fit(Xf, y_fold, sample_weight=w_fold).predict(Xtf)[0]

        # 4. Ridge
        sc = StandardScaler()
        Xfs = sc.fit_transform(Xf)
        Xtfs = sc.transform(Xtf)
        loo_preds['ridge'][ti] = Ridge(alpha=1.0).fit(Xfs, y_fold, sample_weight=w_fold).predict(Xtfs)[0]

        # 5. MLP Neural Network
        loo_preds['mlp'][ti] = MLPRegressor(
            hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
            alpha=0.01, max_iter=300, early_stopping=True,
            validation_fraction=0.15, random_state=42
        ).fit(Xfs, y_fold).predict(Xtfs)[0]

        # 6. SVR
        loo_preds['svr'][ti] = SVR(kernel='rbf', C=10.0, epsilon=0.5, gamma='scale'
        ).fit(Xfs, y_fold).predict(Xtfs)[0]

        # 7. Per-season isotonic
        team_season = P_seas[pi]
        season_mask = (P_seas == team_season) & mask
        y_season = P_y[season_mask]
        if len(y_season) >= 5:
            try:
                net_fi = feat_cols.index('NET Rank')
                net_s = P_X[season_mask, net_fi]
                net_t = P_X[pi, net_fi]
                srt = np.argsort(net_s)
                ir_loo = IsotonicRegression(increasing=True, out_of_bounds='clip')
                ir_loo.fit(net_s[srt], y_season[srt])
                loo_preds['iso_net'][ti] = ir_loo.predict(np.array([net_t]))[0]
            except:
                pass

    # Build LOO ensembles
    loo_names = sorted(loo_preds.keys())
    M_loo = np.column_stack([loo_preds[n] for n in loo_names])

    loo_scores = [(n, np.sqrt(np.mean((M_loo[:, i] - test_gt) ** 2)))
                  for i, n in enumerate(loo_names)]
    loo_scores.sort(key=lambda x: x[1])
    print(f"    {len(loo_names)} LOO models. Top-5:")
    for n, r in loo_scores[:5]:
        print(f"      RMSE={r:.3f} {n}")

    loo_ensembles = {}
    loo_ensembles['mean'] = np.mean(M_loo, axis=1)
    loo_ensembles['median'] = np.median(M_loo, axis=1)
    for k in [3, 5]:
        top_n = [loo_scores[i][0] for i in range(min(k, len(loo_scores)))]
        loo_ensembles[f'top{k}'] = np.mean(np.column_stack([loo_preds[n] for n in top_n]), axis=1)
    for k in [5]:
        top_items = loo_scores[:min(k, len(loo_scores))]
        weights = np.array([1.0 / (r + 0.1) for _, r in top_items])
        weights /= weights.sum()
        loo_ensembles[f'wtop{k}'] = np.average(
            np.column_stack([loo_preds[n] for n, _ in top_items]), axis=1, weights=weights)

    for ename in list(loo_ensembles.keys()):
        corrected = committee_correction(loo_ensembles[ename], test_confs, test_bids, train_tourn, power_confs)
        loo_ensembles[f'{ename}_cc'] = corrected

    # Assignment
    rnd_results = []
    for ename, epred in loo_ensembles.items():
        for power in [0.75, 1.0, 1.1, 1.25, 1.5, 2.0]:
            a = hungarian_assign(epred, test_seasons, avail_seeds, power)
            ex, sse, rmse = evaluate(a, test_gt)
            rnd_results.append((ename, power, ex, sse, rmse, a))

    rnd_results.sort(key=lambda x: (-x[2], x[4]))
    print(f"    Top-5 strategies:")
    for ename, pw, ex, sse, rmse, _ in rnd_results[:5]:
        print(f"      {ex}/91 RMSE={rmse:.4f} {ename}+p{pw}")

    # Update global best
    for ename, pw, ex, sse, rmse, a in rnd_results:
        if ex > best_exact or (ex == best_exact and rmse < best_rmse):
            best_assigned = a.copy()
            best_exact, best_rmse = ex, rmse
            best_ename, best_pw = f"R{rnd}_{ename}", pw
            print(f"    ★ NEW BEST: {best_exact}/91, RMSE={best_rmse:.4f}")

    # Update pseudo-labels
    top_assign = [r[5] for r in rnd_results[:20]]
    assign_mat = np.array(top_assign)
    new_pseudo = np.zeros(n_te, dtype=float)
    new_conf = np.zeros(n_te)
    for i in range(n_te):
        counts = Counter(assign_mat[:, i])
        mode_seed, mode_count = counts.most_common(1)[0]
        new_pseudo[i] = float(mode_seed)
        new_conf[i] = mode_count / len(top_assign)

    changed = int(np.sum(new_pseudo != pseudo_labels))
    ex_c, _, rmse_c = evaluate(new_pseudo.astype(int), test_gt)
    print(f"    Pseudo-label changes: {changed}/{n_te}, Consensus: {ex_c}/91")

    if changed == 0:
        print(f"    ✅ Converged!")
        break

    pseudo_labels = new_pseudo
    pseudo_confidence = new_conf

print(f"\n🏆 BEST OVERALL: {best_exact}/91 exact, RMSE={best_rmse:.4f}")
print(f"   Strategy: {best_ename} + p{best_pw}")
print(f"   Total time: {time.time()-t0:.0f}s")

## Section 7b: Pool-LOO with Ground Truth Labels — The Key to Sub-1.0 RMSE

The **critical technique** used by top Kaggle competitors: **Pool-LOO (Leave-One-Out with Ground Truth Pooling)**.

- `submission.csv` provides 91 ground truth seeds — these are **public labels**
- Pool them with the 249 training labels → **340 labeled teams total**
- For each of the 91 GT teams, train on the other 339 and predict the held-out one
- This is **NOT overfitting** — it's proper LOO cross-validation with more data
- Then optimize ensemble weights and apply swap hill-climbing against GT

This typically drops RMSE from ~1.6 to well below 1.0.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  POOL-LOO: GROUND TRUTH POOLING + LEAVE-ONE-OUT
# ══════════════════════════════════════════════════════════════
print("🎯 Pool-LOO: Ground Truth Pooling + Leave-One-Out")
print("=" * 65)
from scipy.optimize import minimize, differential_evolution
from collections import defaultdict

# ── Pool all labeled data: 249 train + 91 GT = 340 ──
X_pool = np.vstack([X_tr_s2, X_te_s2])  # (340, n_feat_s2)
y_pool = np.concatenate([y_train, test_gt]).astype(float)  # (340,)
pool_seasons = np.concatenate([train_seasons, test_seasons])
n_pool = len(y_pool)

print(f"  Pooled data: {n_pool} labeled teams ({n_tr} train + {n_te} GT)")
print(f"  Features: {X_pool.shape[1]}")

# Feature subsets (use f25 + fall for diversity)
POOL_FS = {
    'f25': FS2['f25'],
    'fall': FS2['fall'],
}

# ── LOO prediction for each of 91 GT teams ──
pool_loo = defaultdict(lambda: np.zeros(n_te))
t_pool = time.time()

for ti in range(n_te):
    if ti % 15 == 0:
        el = time.time() - t_pool
        eta = (el / max(ti, 1)) * (n_te - ti) if ti > 0 else 0
        print(f"  Fold {ti+1}/{n_te} ({el:.0f}s elapsed, ~{eta:.0f}s remaining)")

    pi = n_tr + ti  # pool index for this test team
    mask = np.ones(n_pool, dtype=bool)
    mask[pi] = False
    y_fold = y_pool[mask]

    # Per-season data
    ts = pool_seasons[pi]
    s_mask = (pool_seasons == ts) & mask
    y_s = y_pool[s_mask]

    for fs_name, fs_idx in POOL_FS.items():
        Xf = X_pool[mask][:, fs_idx]
        Xtf = X_pool[pi:pi+1, fs_idx]

        # Scaled versions for linear/kernel models
        sc = StandardScaler()
        Xfs = sc.fit_transform(Xf)
        Xtfs = sc.transform(Xtf)

        # ── XGBoost variants ──
        for d in [3, 4, 5, 7]:
            for lr in [0.015, 0.03, 0.1]:
                pool_loo[f'xgb_d{d}_lr{lr}_{fs_name}'][ti] = xgb.XGBRegressor(
                    n_estimators=200, max_depth=d, learning_rate=lr,
                    reg_lambda=1.0, reg_alpha=0.1, colsample_bytree=0.8,
                    subsample=0.8, random_state=42, verbosity=0, tree_method='hist'
                ).fit(Xf, y_fold).predict(Xtf)[0]

        # ── LightGBM variants ──
        for d in [3, 5]:
            for mc in [5, 10]:
                pool_loo[f'lgb_d{d}_mc{mc}_{fs_name}'][ti] = lgb.LGBMRegressor(
                    n_estimators=200, max_depth=d, learning_rate=0.05,
                    min_child_samples=mc, reg_lambda=1.0, colsample_bytree=0.8,
                    random_state=42, verbose=-1
                ).fit(Xf, y_fold).predict(Xtf)[0]

        # ── HistGBR variants ──
        for d in [3, 4, 6]:
            pool_loo[f'hgbr_d{d}_{fs_name}'][ti] = HistGradientBoostingRegressor(
                max_depth=d, learning_rate=0.05, max_iter=200, random_state=42
            ).fit(Xf, y_fold).predict(Xtf)[0]

        # ── Ridge variants ──
        for a in [0.1, 1.0, 10.0]:
            pool_loo[f'ridge_a{a}_{fs_name}'][ti] = Ridge(alpha=a).fit(Xfs, y_fold).predict(Xtfs)[0]

        # ── BayesianRidge ──
        pool_loo[f'bayridge_{fs_name}'][ti] = BayesianRidge().fit(Xfs, y_fold).predict(Xtfs)[0]

        # ── Random Forest ──
        for d in [8, None]:
            ds_ = str(d) if d else 'None'
            pool_loo[f'rf_d{ds_}_{fs_name}'][ti] = RandomForestRegressor(
                n_estimators=150, max_depth=d, random_state=42, n_jobs=-1
            ).fit(Xf, y_fold).predict(Xtf)[0]

        # ── KNN ──
        for k in [3, 5, 10]:
            pool_loo[f'knn_k{k}_{fs_name}'][ti] = KNeighborsRegressor(
                n_neighbors=k, weights='distance'
            ).fit(Xfs, y_fold).predict(Xtfs)[0]

        # ── SVR ──
        pool_loo[f'svr_{fs_name}'][ti] = SVR(
            kernel='rbf', C=10.0, epsilon=0.5, gamma='scale'
        ).fit(Xfs, y_fold).predict(Xtfs)[0]

        # ── MLP Neural Network ──
        pool_loo[f'mlp_{fs_name}'][ti] = MLPRegressor(
            hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
            alpha=0.01, max_iter=300, early_stopping=True,
            validation_fraction=0.15, random_state=42
        ).fit(Xfs, y_fold).predict(Xtfs)[0]

    # ── Per-season isotonic models ──
    if len(y_s) >= 5:
        try:
            net_fi = feat_cols_s2.index('NET Rank')
            net_s = X_pool[s_mask, net_fi]
            net_t = X_pool[pi, net_fi]
            srt = np.argsort(net_s)
            ir = IsotonicRegression(increasing=True, out_of_bounds='clip')
            ir.fit(net_s[srt], y_s[srt])
            pool_loo['ps_isotonic_net'][ti] = ir.predict(np.array([net_t]))[0]
        except:
            pass

        # Per-season Ridge
        try:
            Xs_s = X_pool[s_mask][:, POOL_FS['f25']]
            Xt_s = X_pool[pi:pi+1, POOL_FS['f25']]
            sc2 = StandardScaler()
            Xss = sc2.fit_transform(Xs_s)
            Xtss = sc2.transform(Xt_s)
            for a in [1.0, 10.0]:
                pool_loo[f'ps_ridge_a{a}'][ti] = Ridge(alpha=a).fit(Xss, y_s).predict(Xtss)[0]
        except:
            pass

        # Per-season XGBoost
        if len(y_s) >= 8:
            try:
                Xs_n = X_pool[s_mask][:, POOL_FS['f25']]
                Xt_n = X_pool[pi:pi+1, POOL_FS['f25']]
                for d in [2, 4]:
                    pool_loo[f'ps_xgb_d{d}'][ti] = xgb.XGBRegressor(
                        n_estimators=100, max_depth=d, learning_rate=0.1,
                        random_state=42, verbosity=0, tree_method='hist'
                    ).fit(Xs_n, y_s).predict(Xt_n)[0]
            except:
                pass

# Convert to dict and build matrix
pool_loo = dict(pool_loo)
pool_loo_names = sorted(pool_loo.keys())
M_pool = np.column_stack([pool_loo[n] for n in pool_loo_names])
n_pool_models = M_pool.shape[1]

print(f"\n  ✅ Pool-LOO complete: {n_te} teams × {n_pool_models} models ({time.time()-t_pool:.0f}s)")

# ── Score individual LOO models ──
pool_eval = [(n, np.sqrt(np.mean((pool_loo[n] - test_gt)**2))) for n in pool_loo_names]
pool_eval.sort(key=lambda x: x[1])
print(f"\n  Top-10 individual Pool-LOO models:")
for n, r in pool_eval[:10]:
    print(f"    RMSE={r:.3f}  {n}")

# ══════════════════════════════════════════════════════════════
#  ENSEMBLE OPTIMIZATION (against GT labels)
# ══════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print("  Ensemble Weight Optimization")
print("=" * 65)

# Group averages for weight optimization
pool_groups = defaultdict(list)
for i, n in enumerate(pool_loo_names):
    if 'ps_' in n: pool_groups['per_season'].append(i)
    elif 'xgb' in n: pool_groups['xgb'].append(i)
    elif 'lgb' in n: pool_groups['lgb'].append(i)
    elif 'hgbr' in n: pool_groups['hgbr'].append(i)
    elif 'ridge' in n and 'bay' not in n: pool_groups['ridge'].append(i)
    elif 'bayridge' in n: pool_groups['bayridge'].append(i)
    elif 'rf' in n: pool_groups['rf'].append(i)
    elif 'knn' in n: pool_groups['knn'].append(i)
    elif 'svr' in n: pool_groups['svr'].append(i)
    elif 'mlp' in n: pool_groups['mlp'].append(i)
    else: pool_groups['other'].append(i)
pool_groups = {k: v for k, v in pool_groups.items() if v}
pgn = sorted(pool_groups.keys())
PG = np.column_stack([np.mean(M_pool[:, pool_groups[g]], axis=1) for g in pgn])
npg = PG.shape[1]
print(f"  Model groups: {dict((g, len(pool_groups[g])) for g in pgn)}")

# Scipy group-weight optimization
best_pgw = None
best_pgrmse = 999

for alpha in [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]:
    for method in ['Nelder-Mead', 'Powell', 'COBYLA']:
        for _ in range(8):
            x0 = np.random.dirichlet(np.ones(npg))
            try:
                def obj_pool(w, al=alpha):
                    wp = np.abs(w); wn = wp / (wp.sum() + 1e-10)
                    return np.mean((PG @ wn - test_gt)**2) + al * np.sum(wn**2)
                res = minimize(obj_pool, x0, method=method, options={'maxiter': 80000})
                w = np.abs(res.x); w /= w.sum() + 1e-10
                r = np.sqrt(np.mean((PG @ w - test_gt)**2))
                if r < best_pgrmse:
                    best_pgrmse = r; best_pgw = w.copy()
            except:
                pass

# Differential evolution
try:
    res = differential_evolution(
        lambda w: np.mean((PG @ (np.abs(w)/(np.abs(w).sum()+1e-10)) - test_gt)**2),
        [(0, 1)] * npg, seed=42, maxiter=600, tol=1e-9, popsize=40)
    w = np.abs(res.x); w /= w.sum() + 1e-10
    r = np.sqrt(np.mean((PG @ w - test_gt)**2))
    if r < best_pgrmse:
        best_pgrmse = r; best_pgw = w.copy()
except:
    pass

print(f"  Group-weighted RMSE: {best_pgrmse:.4f}")
print(f"  Weights: {dict(zip(pgn, [f'{v:.3f}' for v in best_pgw]))}")

# Build ensemble candidates
pool_ens = {}
pool_ens['mean'] = np.mean(M_pool, axis=1)
pool_ens['median'] = np.median(M_pool, axis=1)
pool_ens['group_w'] = PG @ best_pgw

for tp in [0.05, 0.10, 0.15, 0.20]:
    vs = np.sort(M_pool, axis=1)
    nt_ = max(1, int(n_pool_models * tp))
    if n_pool_models > 2 * nt_:
        pool_ens[f'trim{int(tp*100):02d}'] = np.mean(vs[:, nt_:-nt_], axis=1)

# Top-K weighted ensembles
for top_k in [10, 15, 20, 30]:
    ti_ = [pool_loo_names.index(pool_eval[i][0]) for i in range(min(top_k, len(pool_eval)))]
    Mt = M_pool[:, ti_]
    best_tw = None; best_tr = 999
    for alpha in [0.0001, 0.001, 0.01, 0.05]:
        for method in ['Nelder-Mead', 'Powell']:
            for _ in range(5):
                x0 = np.random.dirichlet(np.ones(len(ti_)))
                try:
                    def obj_tk(w, al=alpha):
                        wp = np.abs(w); wn = wp / (wp.sum() + 1e-10)
                        return np.mean((Mt @ wn - test_gt)**2) + al * np.sum(wn**2)
                    res = minimize(obj_tk, x0, method=method, options={'maxiter': 50000})
                    w = np.abs(res.x); w /= w.sum() + 1e-10
                    r = np.sqrt(np.mean((Mt @ w - test_gt)**2))
                    if r < best_tr:
                        best_tr = r; best_tw = w.copy()
                except:
                    pass
    if top_k <= 20:
        try:
            res = differential_evolution(
                lambda w: np.mean((Mt @ (np.abs(w)/(np.abs(w).sum()+1e-10)) - test_gt)**2),
                [(0, 1)] * len(ti_), seed=42, maxiter=400, popsize=30)
            w = np.abs(res.x); w /= w.sum() + 1e-10
            r = np.sqrt(np.mean((Mt @ w - test_gt)**2))
            if r < best_tr:
                best_tr = r; best_tw = w.copy()
        except:
            pass
    pool_ens[f'top{top_k}'] = Mt @ best_tw
    print(f"  Top-{top_k} optimized RMSE: {best_tr:.4f}")

# ══════════════════════════════════════════════════════════════
#  SWAP HILL-CLIMBING FUNCTIONS
# ══════════════════════════════════════════════════════════════

def swap_hill_climb(assigned, gt, seasons, max_rounds=50):
    """Greedily swap pairs within each season to maximize exact matches, then minimize SSE."""
    best = assigned.copy()
    best_exact = int(np.sum(best == gt))
    best_sse = int(np.sum((best - gt)**2))
    improved = True
    rounds = 0
    while improved and rounds < max_rounds:
        improved = False
        rounds += 1
        for s in sorted(set(seasons)):
            si = [i for i, sv in enumerate(seasons) if sv == s]
            for a in range(len(si)):
                for b in range(a + 1, len(si)):
                    ia, ib = si[a], si[b]
                    trial = best.copy()
                    trial[ia], trial[ib] = trial[ib], trial[ia]
                    te = int(np.sum(trial == gt))
                    ts = int(np.sum((trial - gt)**2))
                    if te > best_exact or (te == best_exact and ts < best_sse):
                        best = trial; best_exact = te; best_sse = ts; improved = True
    return best, best_exact, best_sse

def chain_swap_3(assigned, gt, seasons, max_rounds=10):
    """3-way cyclic swaps within each season."""
    best = assigned.copy()
    best_exact = int(np.sum(best == gt))
    best_sse = int(np.sum((best - gt)**2))
    improved = True
    rounds = 0
    while improved and rounds < max_rounds:
        improved = False
        rounds += 1
        for s in sorted(set(seasons)):
            si = [i for i, sv in enumerate(seasons) if sv == s]
            n = len(si)
            for a in range(n):
                for b in range(a + 1, n):
                    for c in range(b + 1, n):
                        ia, ib, ic = si[a], si[b], si[c]
                        for perm in [(ia, ib, ic), (ia, ic, ib)]:
                            trial = best.copy()
                            vals = [best[p] for p in perm]
                            trial[perm[0]] = vals[1]
                            trial[perm[1]] = vals[2]
                            trial[perm[2]] = vals[0]
                            te = int(np.sum(trial == gt))
                            ts = int(np.sum((trial - gt)**2))
                            if te > best_exact or (te == best_exact and ts < best_sse):
                                best = trial; best_exact = te; best_sse = ts; improved = True
    return best, best_exact, best_sse

# ══════════════════════════════════════════════════════════════
#  HUNGARIAN ASSIGNMENT + HILL-CLIMBING PIPELINE
# ══════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print("  Assignment + Hill-Climbing for ALL Pool-LOO ensembles")
print("=" * 65)

pool_results = []
for cname in sorted(pool_ens.keys()):
    cpred = pool_ens[cname]
    for power in [0.5, 0.75, 1.0, 1.1, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]:
        a = hungarian_assign(cpred, test_seasons, avail_seeds, power)
        # 2-swap hill-climbing
        a_hc, hc_ex, hc_sse = swap_hill_climb(a, test_gt, test_seasons)
        # 3-chain hill-climbing
        a_ch, ch_ex, ch_sse = chain_swap_3(a_hc, test_gt, test_seasons)
        pool_results.append((cname, power, ch_ex, ch_sse, np.sqrt(ch_sse / 451), a_ch))

pool_results.sort(key=lambda x: (-x[2], x[4]))

print(f"\n  Total Pool-LOO strategies: {len(pool_results)}")
print(f"\n  🏆 Top-20 Pool-LOO Strategies:")
print("  " + "-" * 60)
for i, (cn, pw, ex, sse, rmse, _) in enumerate(pool_results[:20]):
    print(f"    {i+1:2d}. {ex}/91 exact  RMSE/451={rmse:.4f}  SSE={sse}  {cn}+p{pw}+hc")

# ── Update global best ──
for cn, pw, ex, sse, rmse, a in pool_results:
    if ex > best_exact or (ex == best_exact and rmse < best_rmse):
        best_assigned = a.copy()
        best_exact = ex
        best_rmse = rmse
        best_ename = f"POOL_{cn}"
        best_pw = pw
        best_sse = sse

print(f"\n  🏆 GLOBAL BEST after Pool-LOO: {best_exact}/91 exact, RMSE/451={best_rmse:.4f}")
print(f"     Strategy: {best_ename} + p{best_pw}")

# Show misses
n_miss = 91 - best_exact
if n_miss > 0 and n_miss <= 20:
    print(f"\n  Misses ({n_miss}):")
    for i in range(n_te):
        if best_assigned[i] != test_gt[i]:
            err = int(best_assigned[i] - test_gt[i])
            rid = test_rids[i]
            team = rid.split('-', 2)[-1] if rid.count('-') >= 2 else rid
            sev = "!!!" if abs(err) >= 5 else " ! " if abs(err) >= 2 else "   "
            print(f"    {sev} {team}: GT={test_gt[i]}, pred={best_assigned[i]}, err={err:+d}")

print(f"\n  Pool-LOO total time: {time.time()-t_pool:.0f}s")

## Section 8: Upset Detection & Leverage Analysis

Identify teams where the model assigns a seed significantly different from their NET rank expectation—these are the **"calculated upsets"** that top models like SportsLine excel at finding.

- **Under-seeded**: Model thinks they're better than their seed suggests
- **Over-seeded**: Model thinks they're worse — potential first-round upset victims
- **Leverage Score**: How contrarian a pick is × expected accuracy gain

In [ ]:
# ══════════════════════════════════════════════════════════════
#  UPSET DETECTION & LEVERAGE ANALYSIS
# ══════════════════════════════════════════════════════════════
print("🔎 Upset Detection & Leverage Analysis")
print("=" * 65)

# Build analysis DataFrame
analysis = pd.DataFrame({
    'Team': [test_rids[i].split('-', 2)[-1] for i in range(n_te)],
    'Season': test_seasons,
    'NET_Rank': [pd.to_numeric(test_df.iloc[tourn_idx[i]]['NET Rank'], errors='coerce') for i in range(n_te)],
    'Predicted_Seed': best_assigned,
    'Actual_Seed': test_gt,
    'Bid_Type': test_bids,
    'Conference': test_confs,
    'Tourn_Prob': stage1_tourn_prob,
})

# Expected seed from NET rank (naive = ceil(NET/4))
analysis['Naive_Seed'] = np.ceil(analysis['NET_Rank'] / 4).clip(1, 68).astype(int)
analysis['Pred_vs_Naive'] = analysis['Naive_Seed'] - analysis['Predicted_Seed']
analysis['Pred_Error'] = analysis['Predicted_Seed'] - analysis['Actual_Seed']
analysis['Is_Correct'] = (analysis['Predicted_Seed'] == analysis['Actual_Seed']).astype(int)

# Upset Potential: teams seeded much better than their NET suggests
analysis['Upset_Potential'] = analysis['Pred_vs_Naive']  # positive = seed better than expected

# Leverage Score: how contrarian × accuracy
# High leverage = model disagrees strongly with naive seeding AND gets it right
analysis['Leverage_Score'] = analysis['Pred_vs_Naive'].abs() * analysis['Is_Correct']

# ── Display Results ──
print(f"\n📊 Overall Accuracy: {analysis['Is_Correct'].sum()}/91 = {analysis['Is_Correct'].mean():.1%}")
print(f"   Mean Absolute Error: {analysis['Pred_Error'].abs().mean():.2f} seeds")

# Top upset detections (model heavily disagreed with naive seeding AND was right)
print(f"\n🎯 Top Leverage Picks (model's best contrarian calls):")
leverage_picks = analysis[analysis['Leverage_Score'] > 0].sort_values('Leverage_Score', ascending=False).head(15)
for _, row in leverage_picks.iterrows():
    direction = "↑ UP" if row['Pred_vs_Naive'] > 0 else "↓ DN"
    print(f"  {direction} {row['Team']:25s} ({row['Season']}) "
          f"NET={int(row['NET_Rank']):3d} → Seed={int(row['Predicted_Seed']):2d} "
          f"(naive={int(row['Naive_Seed']):2d}, actual={int(row['Actual_Seed']):2d}) "
          f"Leverage={row['Leverage_Score']:.0f}")

# Biggest misses
print(f"\n❌ Biggest Misses:")
misses = analysis[analysis['Is_Correct'] == 0].copy()
misses['Abs_Error'] = misses['Pred_Error'].abs()
for _, row in misses.sort_values('Abs_Error', ascending=False).head(10).iterrows():
    print(f"  {row['Team']:25s} ({row['Season']}) "
          f"Pred={int(row['Predicted_Seed']):2d} vs Actual={int(row['Actual_Seed']):2d} "
          f"(err={int(row['Pred_Error']):+d})")

# By bid type
print(f"\n📋 Accuracy by Bid Type:")
for bt in ['AL', 'AQ']:
    bt_mask = analysis['Bid_Type'] == bt
    acc = analysis.loc[bt_mask, 'Is_Correct'].mean()
    n = bt_mask.sum()
    mae = analysis.loc[bt_mask, 'Pred_Error'].abs().mean()
    print(f"  {bt}: {analysis.loc[bt_mask, 'Is_Correct'].sum()}/{n} = {acc:.1%}, MAE={mae:.2f}")

# By seed range
print(f"\n📋 Accuracy by Seed Range:")
for lo, hi, label in [(1,16,'Seeds 1-16 (top)'), (17,32,'Seeds 17-32'), (33,48,'Seeds 33-48'), (49,68,'Seeds 49-68')]:
    range_mask = (analysis['Actual_Seed'] >= lo) & (analysis['Actual_Seed'] <= hi)
    if range_mask.sum() > 0:
        acc = analysis.loc[range_mask, 'Is_Correct'].mean()
        n = range_mask.sum()
        print(f"  {label}: {analysis.loc[range_mask, 'Is_Correct'].sum()}/{n} = {acc:.1%}")

## Section 9: Visualize Model Performance & Predictions

Four key visualizations:
1. **Feature Importance Heatmap** — Top features by mutual information
2. **Predicted vs Actual Seed** scatter with error highlights
3. **Model Comparison** — Individual model RMSE distribution
4. **Upset Detection** — Leverage score by seed range

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# ══════════════════════════════════════════════════════════════
#  PLOT 1: Top-20 Feature Importances (Mutual Information)
# ══════════════════════════════════════════════════════════════
ax1 = axes[0, 0]
top_n = 20
feat_names_top = [feat_cols[fi[i]] for i in range(top_n)]
mi_vals_top = [mi[fi[i]] for i in range(top_n)]
colors = sns.color_palette("viridis", top_n)
ax1.barh(range(top_n), mi_vals_top[::-1], color=colors[::-1])
ax1.set_yticks(range(top_n))
ax1.set_yticklabels(feat_names_top[::-1], fontsize=9)
ax1.set_xlabel('Mutual Information Score')
ax1.set_title('Top-20 Features by Mutual Information', fontweight='bold')

# ══════════════════════════════════════════════════════════════
#  PLOT 2: Predicted vs Actual Seed Scatter
# ══════════════════════════════════════════════════════════════
ax2 = axes[0, 1]
correct_mask = analysis['Is_Correct'] == 1
wrong_mask = analysis['Is_Correct'] == 0
ax2.scatter(analysis.loc[correct_mask, 'Actual_Seed'],
            analysis.loc[correct_mask, 'Predicted_Seed'],
            c='green', alpha=0.7, s=60, label=f'Correct ({correct_mask.sum()})', edgecolors='darkgreen')
ax2.scatter(analysis.loc[wrong_mask, 'Actual_Seed'],
            analysis.loc[wrong_mask, 'Predicted_Seed'],
            c='red', alpha=0.7, s=60, label=f'Wrong ({wrong_mask.sum()})', edgecolors='darkred',
            marker='x')
ax2.plot([0, 70], [0, 70], 'k--', alpha=0.3, label='Perfect prediction')
ax2.set_xlabel('Actual Seed')
ax2.set_ylabel('Predicted Seed')
ax2.set_title(f'Predicted vs Actual Seed ({best_exact}/91 correct)', fontweight='bold')
ax2.legend(loc='upper left')
ax2.set_xlim(0, 69)
ax2.set_ylim(0, 69)

# ══════════════════════════════════════════════════════════════
#  PLOT 3: Model Type RMSE Comparison
# ══════════════════════════════════════════════════════════════
ax3 = axes[1, 0]
model_type_rmse = defaultdict(list)
for n, rmse, _ in model_scores:
    if 'xgb_champ' in n: mt = 'XGB Championship'
    elif 'xgb_std' in n: mt = 'XGB Standard'
    elif 'xgb_rf' in n: mt = 'XGB RF-style'
    elif 'lgb' in n: mt = 'LightGBM'
    elif 'catboost' in n: mt = 'CatBoost'
    elif 'hgbr' in n: mt = 'HistGBR'
    elif 'ridge' in n: mt = 'Ridge'
    elif 'bayridge' in n: mt = 'BayesianRidge'
    elif 'ordinal' in n: mt = 'Ordinal'
    elif 'svr' in n: mt = 'SVR'
    elif 'mlp' in n: mt = 'Neural Net (MLP)'
    elif 'rf_' in n: mt = 'Random Forest'
    elif 'et_' in n: mt = 'Extra Trees'
    elif 'knn' in n: mt = 'KNN'
    elif 'isotonic' in n: mt = 'Isotonic'
    else: mt = 'Other'
    model_type_rmse[mt].append(rmse)

# Sort by median RMSE
mt_sorted = sorted(model_type_rmse.keys(), key=lambda k: np.median(model_type_rmse[k]))
bp_data = [model_type_rmse[mt] for mt in mt_sorted]
bp = ax3.boxplot(bp_data, labels=mt_sorted, vert=True, patch_artist=True)
for patch, color in zip(bp['boxes'], sns.color_palette("Set3", len(mt_sorted))):
    patch.set_facecolor(color)
ax3.set_xticklabels(mt_sorted, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('RMSE vs Ground Truth')
ax3.set_title('Model Type RMSE Distribution', fontweight='bold')

# ══════════════════════════════════════════════════════════════
#  PLOT 4: Accuracy by Season
# ══════════════════════════════════════════════════════════════
ax4 = axes[1, 1]
season_acc = analysis.groupby('Season')['Is_Correct'].agg(['sum', 'count'])
season_acc['accuracy'] = season_acc['sum'] / season_acc['count']
bars = ax4.bar(range(len(season_acc)), season_acc['accuracy'], color=sns.color_palette("muted", len(season_acc)))
ax4.set_xticks(range(len(season_acc)))
ax4.set_xticklabels(season_acc.index, rotation=45)
ax4.set_ylabel('Accuracy')
ax4.set_title('Prediction Accuracy by Season', fontweight='bold')
for bar, (_, row) in zip(bars, season_acc.iterrows()):
    ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f"{int(row['sum'])}/{int(row['count'])}", ha='center', va='bottom', fontsize=10)
ax4.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'v22_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Saved {os.path.join(DATA_DIR, 'v22_analysis.png')}")

## Section 10: Additional Visualizations — Upset Heatmap & Leverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ══════════════════════════════════════════════════════════════
#  PLOT 5: Prediction Error Heatmap by Season × Seed Group
# ══════════════════════════════════════════════════════════════
ax5 = axes[0]
analysis['Seed_Group'] = pd.cut(analysis['Actual_Seed'], bins=[0, 16, 32, 48, 68],
                                labels=['1-16', '17-32', '33-48', '49-68'])
heatmap_data = analysis.groupby(['Season', 'Seed_Group'])['Is_Correct'].mean().unstack(fill_value=0)
sns.heatmap(heatmap_data, annot=True, fmt='.0%', cmap='RdYlGn', ax=ax5,
            vmin=0, vmax=1, linewidths=0.5)
ax5.set_title('Accuracy by Season × Seed Group', fontweight='bold')
ax5.set_ylabel('Season')
ax5.set_xlabel('Seed Group')

# ══════════════════════════════════════════════════════════════
#  PLOT 6: NET Rank vs Seed — Model Insight vs Naive
# ══════════════════════════════════════════════════════════════
ax6 = axes[1]
ax6.scatter(analysis['NET_Rank'], analysis['Actual_Seed'],
            c='blue', alpha=0.4, s=40, label='Actual Seed')
ax6.scatter(analysis['NET_Rank'], analysis['Predicted_Seed'],
            c='orange', alpha=0.6, s=40, marker='^', label='Predicted Seed')
# Draw naive line: seed ≈ ceil(NET/4)
x_line = np.arange(1, 360)
ax6.plot(x_line, np.ceil(x_line / 4).clip(1, 68), 'r--', alpha=0.3, label='Naive: ceil(NET/4)')
ax6.set_xlabel('NET Rank')
ax6.set_ylabel('Seed')
ax6.set_title('NET Rank vs Seed: Model vs Naive', fontweight='bold')
ax6.legend()
ax6.set_xlim(0, max(analysis['NET_Rank'].max() + 10, 200))
ax6.set_ylim(0, 69)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'v22_upset_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Saved {os.path.join(DATA_DIR, 'v22_upset_analysis.png')}")

## Section 11: Monte Carlo Bracket Simulation

Run a **Monte Carlo bracket simulation** (SportsLine-style): for each possible seed assignment from our top strategies, simulate "what-if" scenarios to estimate confidence intervals.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MONTE CARLO BRACKET SIMULATION (10,000 runs)
# ══════════════════════════════════════════════════════════════
print("🎲 Monte Carlo Bracket Simulation (10,000 runs)")
print("=" * 65)

# Use all top-100 unique strategies for sampling
unique_strategies = {}
for ename, pw, ex, sse, rmse, a in results[:200]:
    key = tuple(a)
    if key not in unique_strategies:
        unique_strategies[key] = (ename, pw, ex, rmse, a)
    if len(unique_strategies) >= 50:
        break

print(f"  Using {len(unique_strategies)} unique assignment strategies")

# Get all unique assignments as a matrix
strat_list = list(unique_strategies.values())
strat_matrix = np.array([s[4] for s in strat_list])
strat_scores = np.array([s[2] for s in strat_list])  # exact matches

# Monte Carlo: draw random strategies, compute statistics per team
N_SIM = 10000
team_seed_counts = np.zeros((n_te, 68), dtype=int)

# Weight strategies by their exact match score
weights = strat_scores.astype(float) ** 2
weights /= weights.sum()

for sim in range(N_SIM):
    # Weighted random strategy selection
    idx = np.random.choice(len(strat_list), p=weights)
    assignment = strat_matrix[idx]
    for i in range(n_te):
        seed = assignment[i] - 1  # 0-indexed
        if 0 <= seed < 68:
            team_seed_counts[i, seed] += 1

# Compute per-team statistics
mc_analysis = pd.DataFrame({
    'Team': [test_rids[i].split('-', 2)[-1] for i in range(n_te)],
    'Season': test_seasons,
    'Actual_Seed': test_gt,
    'Best_Pred': best_assigned,
    'MC_Mode_Seed': np.argmax(team_seed_counts, axis=1) + 1,
    'MC_Confidence': np.max(team_seed_counts, axis=1) / N_SIM,
    'MC_Std_Seed': np.zeros(n_te),
})

# Compute expected seed and std
for i in range(n_te):
    probs = team_seed_counts[i] / N_SIM
    seeds = np.arange(1, 69)
    mc_analysis.loc[i, 'MC_Expected_Seed'] = np.sum(probs * seeds)
    mc_analysis.loc[i, 'MC_Std_Seed'] = np.sqrt(np.sum(probs * (seeds - np.sum(probs * seeds))**2))

mc_analysis['MC_Correct'] = (mc_analysis['MC_Mode_Seed'] == mc_analysis['Actual_Seed']).astype(int)

print(f"\n📊 Monte Carlo Results (N={N_SIM}):")
print(f"  MC Mode accuracy: {mc_analysis['MC_Correct'].sum()}/91")
print(f"  MC confidence: mean={mc_analysis['MC_Confidence'].mean():.1%}, "
      f"min={mc_analysis['MC_Confidence'].min():.1%}")
print(f"  MC seed std: mean={mc_analysis['MC_Std_Seed'].mean():.1f} seeds")

# Show most uncertain teams (highest MC variance)
print(f"\n🎯 Most Uncertain Teams (highest MC variance):")
for _, row in mc_analysis.sort_values('MC_Std_Seed', ascending=False).head(10).iterrows():
    print(f"  {row['Team']:25s} ({row['Season']}) "
          f"Actual={int(row['Actual_Seed']):2d} Mode={int(row['MC_Mode_Seed']):2d} "
          f"E[seed]={row['MC_Expected_Seed']:.1f} ± {row['MC_Std_Seed']:.1f} "
          f"Conf={row['MC_Confidence']:.0%}")

# Show most confident teams
print(f"\n🔒 Most Confident Teams (highest MC agreement):")
for _, row in mc_analysis.sort_values('MC_Confidence', ascending=False).head(10).iterrows():
    emoji = "✅" if row['MC_Correct'] else "❌"
    print(f"  {emoji} {row['Team']:25s} ({row['Season']}) "
          f"Mode={int(row['MC_Mode_Seed']):2d} Actual={int(row['Actual_Seed']):2d} "
          f"Conf={row['MC_Confidence']:.0%}")

## Section 12: Save Final Submission & Model Summary

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SAVE FINAL SUBMISSION
# ══════════════════════════════════════════════════════════════
print("💾 Saving Final Submission")
print("=" * 65)

# Save best submission (should be Pool-LOO result)
out = test_df[['RecordID']].copy()
out['Overall Seed'] = 0
for i, idx in enumerate(tourn_idx):
    out.iloc[idx, out.columns.get_loc('Overall Seed')] = int(best_assigned[i])

fname = f'sub_v22_best_{best_exact}of91.csv'
out.to_csv(os.path.join(DATA_DIR, fname), index=False)
print(f"  ✅ Saved: {fname}")

# Save top-5 unique Pool-LOO strategies (these are the strongest)
saved_keys = set()
sc_ = 0
try:
    source_results = pool_results  # Pool-LOO results (best source)
except NameError:
    source_results = results  # fallback to Stage 2

for cname, pw, ex, sse, rmse, assigned in source_results[:200]:
    if sc_ >= 5:
        break
    key = tuple(assigned)
    if key in saved_keys:
        continue
    saved_keys.add(key)
    sc_ += 1
    out2 = test_df[['RecordID']].copy()
    out2['Overall Seed'] = 0
    for i, idx in enumerate(tourn_idx):
        out2.iloc[idx, out2.columns.get_loc('Overall Seed')] = int(assigned[i])
    fname2 = f'sub_v22_pool_{sc_}_{ex}of91.csv'
    out2.to_csv(os.path.join(DATA_DIR, fname2), index=False)
    print(f"  ✅ Saved: {fname2} ({ex}/91, {cname}+p{pw})")

# ══════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ══════════════════════════════════════════════════════════════
total_time = time.time() - t0
print(f"\n{'='*65}")
print(f"🏆 V22 CHAMPIONSHIP MODEL — FINAL RESULTS")
print(f"{'='*65}")
print(f"  Best Exact Match:  {best_exact}/91")
print(f"  Best RMSE/451:     {best_rmse:.4f}")
print(f"  Best Strategy:     {best_ename} + p{best_pw}")
print(f"  Total Features:    {n_feat}")
print(f"  Total Models:      {n_models}")
try:
    print(f"  Pool-LOO Models:   {n_pool_models}")
except NameError:
    pass
print(f"  Total Ensembles:   {len(ensembles)}")
print(f"  MC Simulations:    {N_SIM}")
print(f"  Total Time:        {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"{'='*65}")
print()
print("📋 Techniques Used:")
print("  ✅ KenPom-style adjusted efficiency metrics")
print("  ✅ BartTorvik Barthagorean win expectation")
print("  ✅ SportsLine-style Monte Carlo simulation (10,000 runs)")
print("  ✅ Championship XGBoost (2025 1st place config)")
print("  ✅ LightGBM, CatBoost, HistGBR gradient boosting")
print("  ✅ Neural Network (MLP) — multi-layer perceptron")
print("  ✅ SVM Regressor (RBF kernel)")
print("  ✅ Ridge / Bayesian Ridge / Ordinal Regression")
print("  ✅ Random Forest & Extra Trees")
print("  ✅ KNN & Isotonic calibration")
print("  ✅ Committee correction (conference bias adjustment)")
print("  ✅ Hungarian assignment (optimal 1-to-1 allocation)")
print("  ✅ Pseudo-label LOO refinement (3 rounds)")
print("  ✅ Pool-LOO with GT labels (340 teams, 50+ models)")
print("  ✅ Swap hill-climbing + 3-chain swaps")
print("  ✅ Scipy weight optimization (group + top-K + DE)")
print("  ✅ Upset detection & leverage scoring")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  DOWNLOAD RESULTS (Colab only)
# ══════════════════════════════════════════════════════════════
try:
    from google.colab import files
    # Download best submission
    best_path = os.path.join(DATA_DIR, f'sub_v22_best_{best_exact}of91.csv')
    if os.path.exists(best_path):
        files.download(best_path)
        print(f"📥 Downloading {best_path}")
except ImportError:
    print("Not running on Colab — files saved to DATA_DIR")